## Setup

In [1]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import contextily as ctx
import xyzservices as xyz
import matplotlib.pyplot as plt

# import kml reading and set supported driver
import fiona
fiona.drvsupport.supported_drivers["KML"] = "rw"

In [3]:
from gridsample.utils import create_ids, save_shapefiles

In [4]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "00_raw"
PROCESSED_DATA_DIR = DATA_DIR / "01_processed" / "Colonies"

## 1. Load colony shapes

In [5]:
colony_gdf = gpd.read_file(
    RAW_DATA_DIR / "colony_shapes" / "colony_shapes.kml",
    crs="4326",
    driver="KML",
).drop(columns=["Description"])

In [6]:
# remove z-dimension from shapes
colony_gdf.geometry = colony_gdf.geometry.apply(lambda x: shapely.wkb.loads(shapely.wkb.dumps(x, output_dimension=2)))

In [7]:
colony_gdf

,Name,geometry
0,Golden City,"POLYGON ((77.47053 23.16623, 77.47127 23.16765..."
1,Paras Hermitage Society,"POLYGON ((77.45525 23.18311, 77.45564 23.18335..."
2,Eco Green City,"POLYGON ((77.44711 23.27891, 77.44718 23.28040..."
3,Crystal Green Welfare Society,"POLYGON ((77.48964 23.16844, 77.48937 23.16906..."
4,Golden City Prima,"POLYGON ((77.47071 23.17105, 77.47091 23.17141..."


## 2. Load rooftops

In [8]:
from s2cell.s2cell import lat_lon_to_cell_id
import boto3

### Download rooftop data

Get the ID of the level 6 S2 Cell that this colony sits inside

In [9]:
for i, row in colony_gdf.iterrows():
    lat = row.geometry.centroid.y
    lon = row.geometry.centroid.x
    s2_cell_id = lat_lon_to_cell_id(lat=lat, lon=lon, level=6)
    print(f"lat: {lat}, lon: {lon}, s2_cell_id: {s2_cell_id}")
    
s2_cell_id

lat: 23.168474557596397, lon: 77.47040106213896, s2_cell_id: 4142467232250724352
lat: 23.18402985024006, lon: 77.45574937990186, s2_cell_id: 4142467232250724352
lat: 23.2800941829764, lon: 77.44397220719824, s2_cell_id: 4142467232250724352
lat: 23.16886459690637, lon: 77.48810094084455, s2_cell_id: 4142467232250724352
lat: 23.172166438723185, lon: 77.46980105064237, s2_cell_id: 4142467232250724352


4142467232250724352


Download closest S2 cell shapefile from https://beta.source.coop/vida/google-microsoft-open-buildings/geoparquet/by_country_s2/country_iso=IND/

In [10]:
s2_rooftops_path = RAW_DATA_DIR / "rooftops" / f"{s2_cell_id}.parquet"

if s2_rooftops_path.exists():
    print("File already exists")
else:
    s3 = boto3.client("s3", endpoint_url="https://data.source.coop")
    s3.download_file(
        "vida",
        f"google-microsoft-open-buildings/geoparquet/by_country_s2/country_iso=IND/{s2_cell_id}.parquet",
        str(s2_rooftops_path),
    )
    print("File downloaded.")

File already exists


### Load and process rooftop data

In [11]:
rooftop_gdf = gpd.read_parquet(s2_rooftops_path)
# rooftop_gdf.sample(10000).plot(edgecolor="k")

In [12]:
rooftop_gdf = rooftop_gdf.drop(columns=["boundary_id", "s2_id", "geohash", "bbox", "country_iso"])

In [13]:
rooftop_gdf["rooftop_id"] = create_ids(len(rooftop_gdf), f"ROOFTOP_S2_{s2_cell_id}_")

In [14]:
rooftop_gdf

,bf_source,confidence,area_in_meters,geometry,rooftop_id
1312452,google,0.7285,23.7821,"POLYGON ((76.46367 22.05372, 76.46362 22.05372...",ROOFTOP_S2_4142467232250724352_0000001
1515388,google,0.7153,9.8125,"POLYGON ((76.46483 22.05428, 76.46483 22.05432...",ROOFTOP_S2_4142467232250724352_0000002
1710778,google,0.7004,7.0812,"POLYGON ((76.45724 22.05924, 76.45724 22.05927...",ROOFTOP_S2_4142467232250724352_0000003
1252889,microsoft,NaN,68.3447,"POLYGON ((76.45754 22.06020, 76.45746 22.06020...",ROOFTOP_S2_4142467232250724352_0000004
1251238,microsoft,NaN,40.9520,"POLYGON ((76.46078 22.05540, 76.46086 22.05541...",ROOFTOP_S2_4142467232250724352_0000005
...,...,...,...,...,...
566629,microsoft,NaN,25.4265,"POLYGON ((77.91525 23.66157, 77.91520 23.66158...",ROOFTOP_S2_4142467232250724352_2793461
567977,google,0.7441,29.2368,"POLYGON ((77.91905 23.66114, 77.91899 23.66116...",ROOFTOP_S2_4142467232250724352_2793462
567059,google,0.8234,159.8137,"POLYGON ((77.91913 23.66124, 77.91902 23.66128...",ROOFTOP_S2_4142467232250724352_2793463
566685,microsoft,NaN,25.7559,"POLYGON ((77.91606 23.66377, 77.91600 23.66377...",ROOFTOP_S2_4142467232250724352_2793464


## 3. Match rooftops to colonies

In [15]:
subset_rooftops_gdf = rooftop_gdf.sjoin(colony_gdf, how="inner", predicate="intersects").drop(columns=["index_right"])
subset_rooftops_gdf

,bf_source,confidence,area_in_meters,geometry,rooftop_id,Name
909972,google,0.6905,10.6214,"POLYGON ((77.46925 23.16616, 77.46926 23.16619...",ROOFTOP_S2_4142467232250724352_2015004,Golden City
975289,google,0.7298,54.9672,"POLYGON ((77.46939 23.16629, 77.46941 23.16636...",ROOFTOP_S2_4142467232250724352_2015005,Golden City
924952,google,0.8386,75.6909,"POLYGON ((77.46932 23.16617, 77.46933 23.16622...",ROOFTOP_S2_4142467232250724352_2015006,Golden City
856910,google,0.7916,75.5358,"POLYGON ((77.46934 23.16639, 77.46926 23.16641...",ROOFTOP_S2_4142467232250724352_2015007,Golden City
620462,google,0.7688,100.8170,"POLYGON ((77.46920 23.16620, 77.46928 23.16618...",ROOFTOP_S2_4142467232250724352_2015008,Golden City
...,...,...,...,...,...,...
2053633,google,0.7727,134.1622,"POLYGON ((77.44626 23.28045, 77.44614 23.28047...",ROOFTOP_S2_4142467232250724352_2397018,Eco Green City
1946820,google,0.7583,167.9403,"POLYGON ((77.44707 23.28028, 77.44696 23.28031...",ROOFTOP_S2_4142467232250724352_2397020,Eco Green City
2128628,google,0.7469,88.3859,"POLYGON ((77.44696 23.28028, 77.44688 23.28030...",ROOFTOP_S2_4142467232250724352_2397021,Eco Green City
1915569,google,0.8003,139.3598,"POLYGON ((77.44716 23.28026, 77.44707 23.28028...",ROOFTOP_S2_4142467232250724352_2397023,Eco Green City


### Produce plots and save rooftops to file per colony (to be edited manually on MyMaps)

In [16]:
for colony in colony_gdf["Name"].unique():
    folder_path = PROCESSED_DATA_DIR / colony
    folder_path.mkdir(parents=True, exist_ok=True)

    colony_rooftops = subset_rooftops_gdf[subset_rooftops_gdf["Name"] == colony]
    ax = colony_rooftops.plot(figsize=(10, 15))
    # ctx.add_basemap(ax, crs=subset_rooftops_gdf.crs.to_string(), source=xyz.TileProvider.from_qms("Google Satellite Hybrid"))
    ax.set_axis_off()
    plt.title(f"{colony} Rooftops")
    colony_gdf[colony_gdf["Name"] == colony].boundary.plot(ax=ax, color="k")
    plt.savefig(folder_path / "raw_rooftops.png", bbox_inches="tight")
    plt.close()

    save_shapefiles(
        colony_rooftops,
        folder_path,
        f"raw {colony} rooftops",
        formats=["parquet", "kml"],
    )

    ## Get rooftop stats
    colony_rooftops["area_in_meters"].hist(bins=50)
    plt.xlabel("Rooftop area (m2)")
    plt.ylabel("Number of rooftops")
    plt.title(f"{colony} Rooftop Area Distribution")
    plt.savefig(folder_path / "raw_rooftop_area_distribution.png")
    plt.close()

At this point, upload the per-colony rooftops KML to Google MyMaps and edit and correct the shapes, then download the edited KMLs and continue here.

## 4. Load edited rooftops and run analysis

### Load edited rooftop shape KMLs

In [23]:
temp_colony_rooftops_gdf_list = []
for filename in [
    "Crystal_Green_Welfare_Society_rooftops.kml",
    "Golden_City_rooftops.kml",
    "Golden_City_Prima_rooftops.kml",
    "Paras_Hermitage_Society_rooftops.kml",
    "Eco_Green_City_rooftops.kml"
]:
    temp_colony_rooftops_gdf = gpd.read_file(
        RAW_DATA_DIR / "mymaps_edited_colony_rooftops" / filename,
        crs="4326",
        driver="KML",
    ).drop(columns=["Description"])

    temp_colony_rooftops_gdf_list.append(temp_colony_rooftops_gdf)

edited_colony_rooftops_gdf = pd.concat(temp_colony_rooftops_gdf_list, ignore_index=True)
edited_colony_rooftops_gdf.geometry = edited_colony_rooftops_gdf.geometry.apply(
    lambda x: shapely.wkb.loads(shapely.wkb.dumps(x, output_dimension=2))
)

In [24]:
edited_rooftop_areas = edited_colony_rooftops_gdf.to_crs("24378").area
edited_colony_rooftops_gdf["rooftop_area_m2"] = edited_rooftop_areas
edited_colony_rooftops_gdf["rooftop_id"] = create_ids(len(edited_colony_rooftops_gdf), "ROOFTOP_")

In [25]:
# CANCELLED # divide area by 2 for Eco Green City and Crystal Green Welfare Society (since the shapes are for 2 neighbouring buildings and building count will be doubled too)
# edited_colony_rooftops_gdf.loc[
#     edited_colony_rooftops_gdf["Name"].isin(
#         ["Eco Green City", "Crystal Green Welfare Society"]
#     ),
#     "rooftop_area_m2",
# ] /= 2

In [26]:
edited_colony_rooftops_gdf

,Name,geometry,rooftop_area_m2,rooftop_id
0,Crystal Green Welfare Society,"POLYGON ((77.48744 23.16823, 77.48743 23.16835...",159.976012,ROOFTOP_001
1,Crystal Green Welfare Society,"POLYGON ((77.48745 23.16848, 77.48737 23.16848...",177.334952,ROOFTOP_002
2,Crystal Green Welfare Society,"POLYGON ((77.48745 23.16848, 77.48745 23.16857...",122.439256,ROOFTOP_003
3,Crystal Green Welfare Society,"POLYGON ((77.48744 23.16857, 77.48745 23.16870...",173.335860,ROOFTOP_004
4,Crystal Green Welfare Society,"POLYGON ((77.48744 23.16870, 77.48744 23.16881...",125.139260,ROOFTOP_005
...,...,...,...,...
355,Eco Green City,"POLYGON ((77.44414 23.28020, 77.44412 23.28036...",253.018684,ROOFTOP_356
356,Eco Green City,"POLYGON ((77.44410 23.28053, 77.44410 23.28062...",186.983806,ROOFTOP_357
357,Eco Green City,"POLYGON ((77.44412 23.28036, 77.44410 23.28053...",218.650185,ROOFTOP_358
358,Eco Green City,"POLYGON ((77.44398 23.28067, 77.44411 23.28068...",224.810509,ROOFTOP_359


### Run stats

In [27]:
edited_colony_rooftops_gdf["rooftop_area_category"] = pd.cut(
    edited_colony_rooftops_gdf["rooftop_area_m2"],
    bins=[
        0,
        20,
        50,
        100,
        200,
        np.inf,
    ],
    labels=[
        "Rooftop count <20m2",
        "Rooftop count 20-50m2",
        "Rooftop count 50-100m2",
        "Rooftop count 100-200m2",
        "Rooftop count >200m2",
    ],
)

### Save new plots and shapefiles

In [22]:
for colony in [
    "Golden City",
    "Golden City Prima",
    "Paras Hermitage Society",
    "Eco Green City",
    "Crystal Green Welfare Society",
]:
    folder_path = PROCESSED_DATA_DIR / colony
    folder_path.mkdir(parents=True, exist_ok=True)

    colony_rooftops = edited_colony_rooftops_gdf[
        edited_colony_rooftops_gdf["Name"] == colony
    ]
    ax = colony_rooftops.plot(figsize=(10, 15))
    ax.set_axis_off()
    plt.title(f"{colony} Rooftops")
    colony_gdf[colony_gdf["Name"] == colony].boundary.plot(ax=ax, color="k")
    plt.savefig(folder_path / "edited_colony_rooftops.png", bbox_inches="tight")
    plt.close()

    save_shapefiles(
        colony_rooftops,
        folder_path,
        f"edited {colony} rooftops",
        formats=["parquet"],
    )

    ## Get rooftop stats
    colony_rooftops["rooftop_area_m2"].hist(bins=50)
    plt.xlabel("Rooftop area (m2)")
    plt.ylabel("Number of rooftops")
    plt.title(f"{colony} Rooftop Area Distribution")
    plt.savefig(folder_path / "edited_rooftop_area_distribution.png")
    plt.close()

### Pivot to get colony-per-row dataset

In [43]:
pivot_df = edited_colony_rooftops_gdf.pivot_table(index='Name', columns='rooftop_area_category', aggfunc='size', fill_value=0)
pivot_df = pivot_df.reset_index()
pivot_df

rooftop_area_category,Name,Rooftop count <20m2,Rooftop count 20-50m2,Rooftop count 50-100m2,Rooftop count 100-200m2,Rooftop count >200m2
0,Crystal Green Welfare Society,0,2,4,37,7
1,Eco Green City,1,1,6,76,28
2,Golden City,4,0,7,105,19
3,Golden City Prima,0,33,13,1,0
4,Paras Hermitage Society,2,1,3,1,9


In [44]:
# CANCELLED # double the counts for Eco Green City and Crystal Green Welfare Society
# pivot_df.loc[
#     pivot_df["Name"].isin(["Eco Green City", "Crystal Green Welfare Society"]),
#     pivot_df.columns[1:],
# ] *= 2

In [57]:
totals_df = (
    edited_colony_rooftops_gdf.groupby("Name")
    .agg(
        total_rooftop_area_m2=("rooftop_area_m2", "sum"),
        total_rooftop_count=("Name", "size"),
    )
    .reset_index()
)

,Name,total_rooftop_area_m2,total_rooftop_count
0,Crystal Green Welfare Society,9249.316617,50
1,Eco Green City,19311.139455,112
2,Golden City,21606.294511,135
3,Golden City Prima,2403.229120,47
4,Paras Hermitage Society,5244.307948,16


In [59]:
totals_df = pivot_df.merge(totals_df, on="Name")

## 5. Incorporate solar datasets

In [60]:
import rasterio
from rasterio.plot import show
from rasterstats import zonal_stats

In [61]:
solar_data_folderpath = (
    RAW_DATA_DIR
    / "solar_atlas"
    / "India_GISdata_LTAy_YearlyMonthlyTotals_GlobalSolarAtlas-v2_GEOTIFF"
)

### GTI

In [62]:
solar_irradiation_filenames = [
    "GTI.tif",
    "DIF.tif",
    "DNI.tif",
    "GHI.tif",
]
solar_irradiation_column_names = [
    "total_yearly_optimum_tilt_irradiation_kwh",
    "total_yearly_diffuse_irradiation_kwh",
    "total_yearly_direct_normal_irradiation_kwh",
    "total_yearly_horizontal_irradiation_kwh",
]

In [68]:
solar_stats = []
for filename, column_name in zip(solar_irradiation_filenames, solar_irradiation_column_names):

    raster = rasterio.open(solar_data_folderpath / filename)

    # # Plot
    # fig, ax = plt.subplots(figsize=(10, 10))

    # show(raster, ax=ax, title="GeoTIFF Raster")
    # subset_rooftops_gdf.plot(ax=ax, color="r")

    # # restrict the extent of the plot to the extent of the shapes
    # # ax.set_xlim(subset_rooftops_gdf.total_bounds[[0, 2]])
    # # ax.set_ylim(subset_rooftops_gdf.total_bounds[[1, 3]])
    # plt.show()

    # Prep data
    array = raster.read(1)
    affine = raster.transform
    # get per-rooftop irradiation
    solar_stats = zonal_stats(edited_colony_rooftops_gdf, array, affine=affine, nodata=raster.nodata, stats=['sum'], all_touched=True)
    # add to rooftops dataset
    edited_colony_rooftops_gdf[column_name] = [d["sum"] for d in solar_stats]
    # get per-colony stat
    irradiation_colony_totals = edited_colony_rooftops_gdf.groupby("Name")[column_name].sum()
    totals_df[column_name] = totals_df["Name"].map(irradiation_colony_totals)

In [70]:
totals_df = totals_df.round(0)

In [71]:
totals_df

,Name,Rooftop count <20m2,Rooftop count 20-50m2,Rooftop count 50-100m2,Rooftop count 100-200m2,Rooftop count >200m2,total_rooftop_area_m2,total_rooftop_count,total_yearly_optimum_tilt_irradiation_kwh,total_yearly_diffuse_irradiation_kwh,total_yearly_direct_normal_irradiation_kwh,total_yearly_horizontal_irradiation_kwh
0,Crystal Green Welfare Society,0,2,4,37,7,9249.0,50,104092.0,43775.0,76228.0,95604.0
1,Eco Green City,1,1,6,76,28,19311.0,112,278137.0,116272.0,204731.0,255646.0
2,Golden City,4,0,7,105,19,21606.0,135,336821.0,141649.0,246666.0,309443.0
3,Golden City Prima,0,33,13,1,0,2403.0,47,110234.0,46320.0,80761.0,101256.0
4,Paras Hermitage Society,2,1,3,1,9,5244.0,16,35361.0,14821.0,25942.0,32474.0


### 

In [72]:
totals_df.to_csv(PROCESSED_DATA_DIR / "edited_colony_stats.csv", index=False)